In [1]:
import os
import yaml
import copy

import numpy as np
import pandas as pd
import xarray as xr

In [2]:
import xesmf as xe

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

### Fairbanks: TMAX

In [4]:
key = 'Fairbanks'
base_dir = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{key}/'

In [6]:
fn_CESM = base_dir + 'T2_max.zarr'
fn_ERA5 = base_dir + 'ERA5_T2_max.zarr'

ds_CESM = xr.open_zarr(fn_CESM)
ds_ERA5 = xr.open_zarr(fn_ERA5)

In [7]:
ds_CESM = ds_CESM.rename({'init_time': 'year_valid'})
ds_ERA5 = ds_ERA5.rename({'year': 'year_valid'})
varnames = list(ds_ERA5.keys())

ds_collection = []

for i_lead in range(10):
    ds_CESM_slice = ds_CESM.isel(year=i_lead)
    # convert ini_time to valid_time by adding lead times
    ds_CESM_slice['year_valid'] = ds_CESM_slice['year_valid'] + i_lead

    # align two datasets
    ds_ERA5_slice = ds_ERA5
    ds_CESM_slice, ds_ERA5_slice = xr.align(ds_CESM_slice, ds_ERA5_slice, join="inner")

    # compute ACC if there's enough time values
    if len(ds_CESM_slice['year_valid']) >= 10:
        da_collection = []
        for varname in varnames:
            da_collection.append(
                xr.corr(ds_CESM_slice[varname], ds_ERA5_slice[varname], dim="year_valid")
            )
            
        ds_ACC = xr.merge(da_collection)
        ds_collection.append(ds_ACC)

ds_ACC_all = xr.concat(ds_collection, dim='year_valid')

In [10]:
save_name = base_dir + 'ACC_TMAX.zarr'
ds_ACC_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/Fairbanks/ACC_TMAX.zarr


In [11]:
ds = xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/METRICS/Fairbanks/ACC_TMAX.zarr')

In [15]:
# ds['TREFHTMX_max'].values